# Delta Lake Management and Streaming

## Business Context

Welcome to the **NYC Yellow Taxi Analytics Platform**! As a Data Engineer at MetroFleet Analytics, you're supporting NYC's largest taxi dispatch network. Your fleet processes thousands of rides daily across 200+ zones, and operations managers need reliable, real-time insights into:

- **Zone Performance**: Which pickup/dropoff zones generate the most revenue?
- **Fare Trends**: How do trip distances and fares vary throughout the day?
- **Data Quality**: Managing corrections to fare data (cancellations, adjustments, duplicate trips)
- **Real-Time Operations**: Streaming new trip completions as they happen

Previously, your data warehouse ran batch updates every 6 hours, causing stale analytics and missed optimization opportunities. You'll build a modern **Delta Lake pipeline** that enables efficient data management, optimized query performance, and real-time ingestion.

## Dataset Overview

You'll work with NYC Yellow Taxi trip data from `samples.nyctaxi.trips`:

| Column | Type | Description | Usage |
|--------|------|-------------|-------|
| `tpep_pickup_datetime` | timestamp | When passenger was picked up | Time-based analytics, windowing |
| `tpep_dropoff_datetime` | timestamp | When passenger was dropped off | Trip duration calculation |
| `trip_distance` | double | Trip distance in miles | Revenue per mile analysis |
| `fare_amount` | double | Fare charged in USD | Revenue aggregations |
| `pickup_zip` | int | Pickup location ZIP code | Zone performance, Z-order optimization |
| `dropoff_zip` | int | Dropoff location ZIP code | Route analysis |

**Sample Size**: ~100,000 trips from NYC Yellow Taxi dataset

## Learning Objectives

This comprehensive lab covers three core Delta Lake topics compatible with Databricks Free Edition:

1. **Delta Lake Foundations** - Write to Delta format, schema evolution, time travel, table history
2. **Data Management Operations** - CRUD operations (INSERT/UPDATE/DELETE), MERGE upserts, OPTIMIZE with Z-ordering, VACUUM
3. **Streaming Pipeline with Auto Loader** - Incremental file ingestion, structured streaming, windowed aggregations, streaming MERGE

**Note**: This lab uses `trigger(availableNow=True)` for batch-style processing compatible with Databricks Free Edition serverless compute. The Delta Lake and Structured Streaming APIs you'll learn are identical to production environments!

## Lab Journey

**Act 1: Establish the Data Lake** → Convert raw taxi data to Delta format with proper schema management
**Act 2: Manage Data Quality** → Handle corrections, deletions, and incremental updates efficiently
**Act 3: Real-Time Ingestion** → Build Bronze → Silver → Gold streaming pipeline with Auto Loader

## Setup

In [0]:
%sql
-- Create a catalog and schema for our Delta Lake lab
CREATE CATALOG IF NOT EXISTS nyctaxi_catalog;

-- Create a schema (database) in the catalog
CREATE SCHEMA IF NOT EXISTS nyctaxi_catalog.analytics;

-- Create a managed volume for file storage
CREATE VOLUME IF NOT EXISTS nyctaxi_catalog.analytics.workspace;

In [0]:
# Set up working directory using Unity Catalog volume
import os

# Use Unity Catalog managed volume for file storage
working_dir = "/Volumes/nyctaxi_catalog/analytics/workspace"
checkpoint_dir = f"{working_dir}/checkpoints"

print(f"Working directory: {working_dir}")
print(f"Checkpoint directory: {checkpoint_dir}")

Working directory: /Volumes/nyctaxi_catalog/analytics/workspace
Checkpoint directory: /Volumes/nyctaxi_catalog/analytics/workspace/checkpoints


In [0]:
# Clean up working directory to account for any failed previous runs.
# IMPORTANT: Stop any active streaming queries first
for query in spark.streams.active:
    print(f"Stopping query: {query.id}")
    query.stop()

# Now clean up directories
try:
    dbutils.fs.rm(f"{working_dir}", recurse=True)
    print(f"✅ Cleaned up working directory: {working_dir}")
except Exception as e:
    print(f"⚠️ Cleanup note: {e}")
    print("Continuing with lab...")

# Recreate working directory structure
dbutils.fs.mkdirs(working_dir)
dbutils.fs.mkdirs(checkpoint_dir)

✅ Cleaned up working directory: /Volumes/nyctaxi_catalog/analytics/workspace


True

---
# Important: Databricks Free Edition Compatibility

This lab is designed for **Databricks Free Edition** with serverless compute, which has specific limitations:

## Streaming Limitations:
- ❌ **Cannot use `display()` on streaming DataFrames** - Interactive streaming display not supported
- ❌ **No continuous streaming** - Cannot use default or time-based triggers
- ❌ **"update" output mode not supported** - Use "append" for windowed aggregations and "complete" for non-windowed aggregations
- ✅ **Use `trigger(availableNow=True)`** - Processes all available data in batch mode
- ✅ **Use watermarks with aggregations** - Required for append mode to work with windowed aggregations
- ✅ **Verify outputs** - Read streaming results as batch DataFrames for verification

## Delta Lake Features:
- ✅ **All core Delta Lake features supported** - CRUD, MERGE, OPTIMIZE, Z-order, VACUUM, time travel
- ✅ **Auto Loader (cloudFiles) supported** - Schema inference and incremental ingestion

## Pattern Used Throughout This Lab:

1. **Create streaming DataFrame** with `readStream`
2. **Apply transformations** (filters, aggregations, joins)
3. **Write with `writeStream`** using `trigger(availableNow=True)`
4. **Verify by reading output** as batch DataFrame and using `display()`

This approach simulates real-time streaming while working within Free Edition constraints. You're learning the exact same APIs used in production!

---
# Section 1: Delta Lake Foundations

**Business Goal:** Establish a reliable Delta Lake table with proper schema management and versioning for historical trip analysis.

In this section, you'll learn how to work with Delta Lake tables: writing data in Delta format, evolving schemas to add new columns, handling breaking schema changes, and using time travel to query historical versions.

## Key Concepts:
- **Delta Format**: ACID-compliant storage format built on Parquet
- **Schema Evolution**: Adding new columns without rewriting entire dataset
- **Schema Enforcement**: Preventing incompatible data types
- **overwriteSchema**: Enabling breaking schema changes (type modifications)
- **Time Travel**: Querying previous versions of the table
- **Table History**: Audit trail of all table changes

### Task 1.1: Write Initial Delta Table

Create your baseline trip history by converting NYC Yellow Taxi data from the `samples` catalog into Delta format. Delta Lake provides ACID transactions, scalable metadata handling, and time travel capabilities built on top of Parquet files.

**Why Delta?** Traditional Parquet files don't support updates, deletes, or schema evolution. Delta Lake adds these critical features while maintaining compatibility with existing Spark DataFrames.

In [0]:
# TODO: Load NYC taxi trips from samples catalog and write to Delta format
# Use spark.table() to load, then write with Delta format

from pyspark.sql.functions import col

# Load taxi trips
trips_df = spark.table("samples.nyctaxi.trips")  # Table name from samples catalog

print(f"Loaded {trips_df.count():,} taxi trips")
display(trips_df.limit(10))

# Write to Delta format
(trips_df
 .write
 .format("delta")  # Delta format
 .mode("overwrite")  # Write mode
 .save(f"{working_dir}/taxi_trips_delta")
)

print(f"✅ Written trips to Delta format at {working_dir}/taxi_trips_delta")

Loaded 21,932 taxi trips


tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022
2016-02-13T00:41:43.000Z,2016-02-13T00:46:52.000Z,1.4,6.5,10023,10069
2016-02-18T23:49:53.000Z,2016-02-19T00:12:53.000Z,10.4,31.0,11371,10003
2016-02-18T20:21:45.000Z,2016-02-18T20:38:23.000Z,10.15,28.5,11371,11201
2016-02-03T10:47:50.000Z,2016-02-03T11:07:06.000Z,3.27,15.0,10014,10023
2016-02-19T01:26:39.000Z,2016-02-19T01:40:01.000Z,4.42,15.0,10003,11222


✅ Written trips to Delta format at /Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta


In [0]:
# CHECK YOUR WORK
delta_df = spark.read.format("delta").load(f"{working_dir}/taxi_trips_delta")
assert delta_df.count() > 0, "Delta table should contain data"
assert set(delta_df.columns) == {"tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "fare_amount", "pickup_zip", "dropoff_zip"}, "Should have correct columns"
print("✅ Task 1.1 complete: Initial Delta table created")

✅ Task 1.1 complete: Initial Delta table created


### Task 1.2: Schema Evolution - Add Calculated Columns

Operations managers need additional metrics: trip duration (minutes) and average speed (mph). Add these as calculated columns using **schema evolution**.

**Challenge**: By default, Delta Lake enforces schema-on-write - new columns in appended data will cause errors. You'll need to enable schema evolution with `.option("mergeSchema", "true")`.

In [0]:
# TODO: Add trip_duration_minutes and avg_speed_mph columns, then append with schema merge
# Calculate duration: (dropoff_time - pickup_time) in minutes
# Calculate speed: trip_distance / (duration_in_hours)

from pyspark.sql.functions import unix_timestamp, round, try_divide

# Read existing Delta table
base_df = spark.read.format("delta").load(f"{working_dir}/taxi_trips_delta")

# Add calculated columns
enhanced_df = base_df.withColumn(
    "trip_duration_minutes",
    round((unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime")))/60, 2)  # Correct column names
).withColumn(
    "avg_speed_mph",
    round(try_divide(col("trip_distance"), col("trip_duration_minutes") / 60), 2)
)

print("Enhanced schema:")
enhanced_df.printSchema()

# Try appending without mergeSchema (will fail!)
try:
    enhanced_df.write.format("delta").mode("append").save(f"{working_dir}/taxi_trips_delta")
    print("❌ This shouldn't succeed without mergeSchema!")
except Exception as e:
    print(f"✅ Expected error: {str(e)[:100]}...")

# Now enable schema evolution
(enhanced_df
 .write
 .format("delta")
 .mode("append")
 .option("mergeSchema", "true")
 .save(f"{working_dir}/taxi_trips_delta")
)

print("✅ Schema evolution successful - new columns added")

Enhanced schema:
root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- avg_speed_mph: double (nullable = true)

✅ Expected error: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID...
✅ Schema evolution successful - new columns added


In [0]:
# CHECK YOUR WORK
evolved_df = spark.read.format("delta").load(f"{working_dir}/taxi_trips_delta")
assert "trip_duration_minutes" in evolved_df.columns, "Should have trip_duration_minutes column"
assert "avg_speed_mph" in evolved_df.columns, "Should have avg_speed_mph column"
assert evolved_df.count() >= base_df.count(), "Row count should be at least as large as the original"
print("✅ Task 1.2 complete: Schema evolution with calculated columns")

✅ Task 1.2 complete: Schema evolution with calculated columns


### Task 1.3: Overwrite with Breaking Schema Change

The finance department requires `fare_amount` as `DECIMAL(10,2)` instead of `DOUBLE` for accounting precision. This is a **breaking schema change** - simply adding a column is safe, but changing a column's data type requires `overwriteSchema`.

**Warning**: `overwriteSchema=true` replaces the entire table schema. Use carefully!

In [0]:
# TODO: Cast fare_amount to DECIMAL(10,2) and overwrite with schema change
# This demonstrates a breaking schema change that requires overwriteSchema

from pyspark.sql.types import DecimalType
from pyspark.sql.functions import col

# Read current data (with duplicates from previous append)
current_df = spark.read.format("delta").load(f"{working_dir}/taxi_trips_delta")

# Remove duplicates and cast fare_amount
transformed_df = (current_df
    .dropDuplicates(["tpep_pickup_datetime", "pickup_zip", "dropoff_zip"])
    .withColumn("fare_amount", col("fare_amount").cast(DecimalType(10,2)))  # Cast to DecimalType
)

print("Transformed schema:")
transformed_df.printSchema()

# Overwrite with schema change
(transformed_df
 .write
 .format("delta")  # Delta format
 .mode("overwrite")  # Overwrite mode
 .option("overwriteSchema","true")  # Option for schema overwrite
 .save(f"{working_dir}/taxi_trips_delta")
)

print("✅ Schema overwrite complete - fare_amount is now DECIMAL(10,2)")

Transformed schema:
root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: decimal(10,2) (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- avg_speed_mph: double (nullable = true)

✅ Schema overwrite complete - fare_amount is now DECIMAL(10,2)


In [0]:
# CHECK YOUR WORK
final_df = spark.read.format("delta").load(f"{working_dir}/taxi_trips_delta")
fare_field = [f for f in final_df.schema.fields if f.name == "fare_amount"][0]
assert "decimal" in str(fare_field.dataType).lower(), "fare_amount should be DecimalType"
assert final_df.count() == trips_df.count(), "Should have original row count (duplicates removed)"
print("✅ Task 1.3 complete: Breaking schema change applied")

✅ Task 1.3 complete: Breaking schema change applied


### Task 1.4: Time Travel and Table History

One of Delta Lake's most powerful features is **time travel** - the ability to query previous versions of your table. Every write operation creates a new version, and Delta Lake maintains a transaction log so you can:
- Audit data changes
- Rollback to previous versions
- Reproduce ML training datasets
- Debug data quality issues

Query your table's history and compare Version 0 (original) vs Version 2 (final) to see how the data evolved.

**Use SQL for table history inspection:**

In [0]:
%sql
-- TODO: View complete table history
-- Replace with the path to your Delta table

DESCRIBE HISTORY delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-03-25T16:42:15.000Z,74432780357360,mwaibanjehope@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(3443473830815790),c545cac3-59df-4648-ad53-5afa5a2569b2,0325-163847-uavphepk-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 1005222, numDeletionVectorsRemoved -> 0, numOutputRows -> 21932, numOutputBytes -> 555846)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-03-25T16:42:10.000Z,74432780357360,mwaibanjehope@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3443473830815790),5905b953-bd1d-428d-a343-180e7e8cb28a,0325-163847-uavphepk-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 21932, numOutputBytes -> 548401)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-03-25T16:42:04.000Z,74432780357360,mwaibanjehope@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(3443473830815790),78d8e0d5-f356-41d9-b176-7acf5e8a8180,0325-163847-uavphepk-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 21932, numOutputBytes -> 456821)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


**Query specific versions:**

In [0]:
%sql
-- TODO: Query Version 0 (original data, no calculated columns)
-- Use VERSION AS OF to query a specific version

SELECT * FROM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta` VERSION AS OF 0 LIMIT 10

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022
2016-02-13T00:41:43.000Z,2016-02-13T00:46:52.000Z,1.4,6.5,10023,10069
2016-02-18T23:49:53.000Z,2016-02-19T00:12:53.000Z,10.4,31.0,11371,10003
2016-02-18T20:21:45.000Z,2016-02-18T20:38:23.000Z,10.15,28.5,11371,11201
2016-02-03T10:47:50.000Z,2016-02-03T11:07:06.000Z,3.27,15.0,10014,10023
2016-02-19T01:26:39.000Z,2016-02-19T01:40:01.000Z,4.42,15.0,10003,11222


In [0]:
%sql
-- TODO: Query current version (with calculated columns and DECIMAL fare_amount)
SELECT * FROM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta` LIMIT 10

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip,trip_duration_minutes,avg_speed_mph
2016-02-18T23:49:53.000Z,2016-02-19T00:12:53.000Z,10.4,31.00,11371,10003,23.0,27.13
2016-02-06T23:25:43.000Z,2016-02-06T23:31:08.000Z,1.1,6.00,11237,11237,5.42,12.18
2016-02-20T19:28:12.000Z,2016-02-20T20:08:33.000Z,19.6,54.50,11422,11238,40.35,29.14
2016-02-21T08:58:53.000Z,2016-02-21T09:02:28.000Z,0.7,4.50,10025,10162,3.58,11.73
2016-02-09T15:31:27.000Z,2016-02-09T15:40:16.000Z,1.18,8.00,10026,10025,8.82,8.03
2016-02-09T08:11:51.000Z,2016-02-09T08:19:24.000Z,1.2,6.50,10021,10028,7.55,9.54
2016-02-27T15:55:25.000Z,2016-02-27T16:13:48.000Z,3.18,13.50,10021,10001,18.38,10.38
2016-02-23T13:14:30.000Z,2016-02-23T13:31:39.000Z,4.2,16.50,10018,10282,17.15,14.69
2016-02-19T07:57:23.000Z,2016-02-19T08:01:14.000Z,0.48,4.50,10021,10021,3.85,7.48
2016-02-17T10:48:24.000Z,2016-02-17T10:59:54.000Z,1.1,8.50,10011,10110,11.5,5.74


In [0]:
%sql
-- TODO: Compare average fare between Version 0 and current
SELECT
    'Version 0' AS version,
    COUNT(*) AS trip_count,
    AVG(CAST(fare_amount AS DOUBLE)) AS avg_fare
FROM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta` VERSION AS OF 0

UNION ALL

SELECT
    'Current' AS version,
    COUNT(*) AS trip_count,
    AVG(CAST(fare_amount AS DOUBLE)) AS avg_fare
FROM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta` 

version,trip_count,avg_fare
Version 0,21932,12.348726974284153
Current,21932,12.348726974284153


In [0]:
# CHECK YOUR WORK
# Verify we can read Version 0
version_0_df = spark.read.format("delta").option("versionAsOf", 0).load(f"{working_dir}/taxi_trips_delta")
assert "trip_duration_minutes" not in version_0_df.columns, "Version 0 should not have calculated columns"
print("✅ Task 1.4 complete: Time travel and table history explored")

✅ Task 1.4 complete: Time travel and table history explored


---
# Section 2: Data Management Operations

**Business Goal:** Handle real-world data corrections, deletions, and incremental updates efficiently using Delta Lake's CRUD operations.

In production, data is never static. You need to:
- **INSERT** new batches of trips as they arrive
- **UPDATE** fares when surge pricing is applied retroactively
- **DELETE** cancelled or fraudulent trips
- **MERGE** daily corrections (fare adjustments, duplicate resolution)
- **OPTIMIZE** storage with file compaction and Z-ordering
- **VACUUM** to reclaim storage from old versions

## Key Concepts:
- **ACID Transactions**: All operations are atomic, consistent, isolated, durable
- **MERGE (Upsert)**: Update existing records or insert new ones in a single operation
- **OPTIMIZE**: Compact small files into larger ones for better query performance
- **Z-Order**: Collocate related data in the same files (e.g., group by pickup_zip)
- **VACUUM**: Remove old data files no longer referenced by current table version

### Task 2.1: UPDATE - Apply Surge Pricing

During evening rush hour (5PM-7PM), pickup zone 10001 (Midtown Manhattan) had high demand. Apply a retroactive 10% surge pricing adjustment to these trips.

**Business Impact**: Operations can now correct pricing after analyzing demand patterns, ensuring fair compensation for drivers during peak periods.

In [0]:
%sql
-- TODO: Update fare_amount for surge pricing
-- Conditions: pickup_zip = 10001 AND hour between 17-19 (5PM-7PM)
-- Action: Increase fare by 10% (fare * 1.10)

UPDATE delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`
SET fare_amount = fare_amount * 1.10
WHERE pickup_zip = 10001 AND hour(tpep_pickup_datetime) BETWEEN 17 AND 19

num_affected_rows
221


In [0]:
%sql
-- Verify the update
SELECT pickup_zip, hour(tpep_pickup_datetime) AS pickup_hour,
       COUNT(*) AS trips, AVG(CAST(fare_amount AS DOUBLE)) AS avg_fare
FROM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`
WHERE pickup_zip = 10001
GROUP BY pickup_zip, pickup_hour
ORDER BY pickup_hour;

pickup_zip,pickup_hour,trips,avg_fare
10001,0,57,12.289473684210526
10001,1,31,11.129032258064516
10001,2,25,11.44
10001,3,18,10.86111111111111
10001,4,18,12.694444444444445
10001,5,11,16.59181818181818
10001,6,21,7.976190476190476
10001,7,28,8.017857142857142
10001,8,39,8.679487179487179
10001,9,50,10.08


In [0]:
# CHECK YOUR WORK
updated_trips = spark.sql("""
    SELECT COUNT(*) AS cnt FROM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`
    WHERE pickup_zip = 10001 AND hour(tpep_pickup_datetime) BETWEEN 17 AND 19
""").collect()[0]["cnt"]
print(f"Updated {updated_trips} trips with surge pricing")
print("✅ Task 2.1 complete: Surge pricing applied")

Updated 221 trips with surge pricing
✅ Task 2.1 complete: Surge pricing applied


### Task 2.2: DELETE - Remove Invalid Trips

Data quality audit revealed trips with invalid values:
- Zero or negative fares (payment processing errors)
- Negative trip distances (GPS errors)
- Zero or negative trip duration (timestamp errors)

Remove these records to ensure analytics accuracy.

In [0]:
%sql
-- Count invalid trips before deletion
SELECT 'Invalid Fares' AS issue, COUNT(*) AS count
FROM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`
WHERE fare_amount <= 0

UNION ALL

SELECT 'Invalid Distance' AS issue, COUNT(*) AS count
FROM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`
WHERE trip_distance < 0

UNION ALL

SELECT 'Invalid Duration' AS issue, COUNT(*) AS count
FROM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`
WHERE trip_duration_minutes <= 0;

issue,count
Invalid Fares,10
Invalid Distance,0
Invalid Duration,1


In [0]:
%sql
-- TODO: Delete invalid trips
-- Conditions: fare_amount <= 0 OR trip_distance < 0 OR trip_duration_minutes <= 0

DELETE FROM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`
WHERE fare_amount <= 0 OR trip_distance < 0 OR trip_duration_minutes <= 0;

num_affected_rows
11


In [0]:
%sql
-- Verify deletions
SELECT COUNT(*) AS remaining_trips FROM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`;

remaining_trips
21921


In [0]:
# CHECK YOUR WORK
remaining_df = spark.read.format("delta").load(f"{working_dir}/taxi_trips_delta")
assert remaining_df.filter("fare_amount <= 0").count() == 0, "Should have no invalid fares"
assert remaining_df.filter("trip_distance < 0").count() == 0, "Should have no invalid distances"
print("✅ Task 2.2 complete: Invalid trips removed")

✅ Task 2.2 complete: Invalid trips removed


### Task 2.3: MERGE - Daily Batch Corrections

Each night, the billing system sends a corrections file with:
- **Fare adjustments** (complaints, promotions, refunds)
- **New trips** that were delayed in processing

Use **MERGE** to apply these updates efficiently in a single operation (upsert = update + insert).

In [0]:
# TODO
# Create corrections dataset: apply 5% discount to high-fare trips in zone 10001
corrections_df = spark.sql("""
    SELECT
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        trip_distance,
        CAST(fare_amount * 0.95 AS DECIMAL(10,2)) AS fare_amount,  -- 5% discount
        pickup_zip,
        dropoff_zip,
        trip_duration_minutes,
        avg_speed_mph
    FROM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`
    WHERE pickup_zip = 10001 AND fare_amount > 50
    LIMIT 100
""")

# Also add some "new" trips by modifying timestamps
from pyspark.sql.functions import expr, lit

new_trips_df = spark.sql("""
    SELECT
        tpep_pickup_datetime + INTERVAL 1 DAY AS tpep_pickup_datetime,
        tpep_dropoff_datetime + INTERVAL 1 DAY AS tpep_dropoff_datetime,
        trip_distance,
        CAST(fare_amount AS DECIMAL(10,2)) AS fare_amount,
        pickup_zip,
        dropoff_zip,
        trip_duration_minutes,
        avg_speed_mph
    FROM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`
    WHERE pickup_zip = 10002
    LIMIT 50
""")

# Combine corrections and new trips
all_corrections_df = corrections_df.union(new_trips_df)

print(f"Corrections: {corrections_df.count()} fare adjustments + {new_trips_df.count()} new trips")

# Register as temp view for MERGE
all_corrections_df.createOrReplaceTempView("trip_corrections")

Corrections: 13 fare adjustments + 50 new trips


In [0]:
%sql
-- TODO: MERGE corrections into main table
-- Match on: pickup/dropoff times and pickup_zip
-- When matched: UPDATE fare_amount
-- When not matched: INSERT all columns

MERGE INTO delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta` AS target
USING trip_corrections AS source
ON target.tpep_pickup_datetime = source.tpep_pickup_datetime
   AND target.tpep_dropoff_datetime = source.tpep_dropoff_datetime
   AND target.pickup_zip = source.pickup_zip
WHEN MATCHED THEN
    UPDATE SET target.fare_amount = source.fare_amount
WHEN NOT MATCHED THEN
    INSERT (tpep_pickup_datetime, tpep_dropoff_datetime, trip_distance, fare_amount, pickup_zip, dropoff_zip, trip_duration_minutes, avg_speed_mph)
    VALUES (source.tpep_pickup_datetime, source.tpep_dropoff_datetime, source.trip_distance, source.fare_amount, source.pickup_zip, source.dropoff_zip, source.trip_duration_minutes, source.avg_speed_mph)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
63,13,0,50


In [0]:
%sql
-- Verify MERGE results
SELECT COUNT(*) AS total_trips FROM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`;

total_trips
21971


In [0]:
# CHECK YOUR WORK
print("✅ Task 2.3 complete: Daily corrections merged")

✅ Task 2.3 complete: Daily corrections merged


### Task 2.4: OPTIMIZE with Z-Order

Over time, many small files accumulate (each write creates new Parquet files). This degrades query performance. **OPTIMIZE** compacts small files into larger ones.

Additionally, use **Z-ordering** on `pickup_zip` to collocate data from the same zones together. This dramatically speeds up queries that filter by zone (e.g., "show me all trips from zone 10001").

**Z-Order Intuition**: Instead of random file layout, Z-order clusters related data. Queries filtering by pickup_zip can skip entire files!

In [0]:
%sql
-- Check current file count
DESCRIBE DETAIL delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,df2f771f-1fd5-433a-86bd-9b6036b229f4,null,null,dbfs:/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta,2026-03-25T16:41:53.998Z,2026-03-25T16:43:18.000Z,List(),List(),4,573112,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 245, numDeletionVectors -> 2)",false


In [0]:
%sql
-- TODO: Compact files and Z-order by pickup_zip

OPTIMIZE delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`
ZORDER BY (pickup_zip);

path,metrics
dbfs:/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta,"List(1, 4, List(556552, 556552, 556552.0, 1, 556552), List(2932, 555846, 143278.0, 4, 573112), 0, List(minCubeSize(107374182400), List(0, 0), List(4, 573112), 0, List(4, 573112), 1, null), null, 0, 1, 4, 0, false, 0, 0, 1774457005493, 1774457007911, 8, 1, null, List(2, 245), null, 8, 8, 369, 0, null, null)"


In [0]:
%sql
-- Check file count after optimization
DESCRIBE DETAIL delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta`;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,df2f771f-1fd5-433a-86bd-9b6036b229f4,null,null,dbfs:/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta,2026-03-25T16:41:53.998Z,2026-03-25T16:43:28.000Z,List(),List(),1,556552,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
# CHECK YOUR WORK
print("✅ Task 2.4 complete: Table optimized with Z-ordering")

✅ Task 2.4 complete: Table optimized with Z-ordering


### Task 2.5: VACUUM - Reclaim Storage

OPTIMIZE created new compacted files, but the old small files still exist (for time travel). **VACUUM** permanently deletes files not referenced by the current table version.

**Note**: VACUUM uses a default retention period of 7 days. Files older than this period that are no longer referenced will be deleted. In production, you can specify custom retention periods like `RETAIN 168 HOURS`.

In [0]:
%sql
-- TODO: Preview files that will be deleted (DRY RUN)
-- Note: 168 hours (7 days) is the minimum retention period

VACUUM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta` RETAIN 168 HOURS DRY RUN;

path


In [0]:
%sql
-- TODO: Execute vacuum (permanently delete old files)
-- Note: Files older than 168 hours (7 days) will be deleted

VACUUM delta.`/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta` RETAIN 168 HOURS;

path
dbfs:/Volumes/nyctaxi_catalog/analytics/workspace/taxi_trips_delta


In [0]:
# CHECK YOUR WORK
print("✅ Task 2.5 complete: Old data files vacuumed")
print("⚠️  Note: Files older than the retention period are permanently deleted")

✅ Task 2.5 complete: Old data files vacuumed
⚠️  Note: Files older than the retention period are permanently deleted


---
# Section 3: Streaming Pipeline with Auto Loader

**Business Goal:** Build a real-time trip ingestion pipeline using Auto Loader and Structured Streaming to process new trip files as they arrive.

Modern data architectures use a **Medallion Architecture**:
- **Bronze**: Raw ingestion (Auto Loader) - no transformations, just load files incrementally
- **Silver**: Cleaned and aggregated data - business logic applied
- **Gold**: Business-level aggregates - ready for analytics and BI tools

You'll build all three layers with NYC taxi trip data.

## Key Concepts:
- **Auto Loader (cloudFiles)**: Incrementally load new files with automatic schema inference
- **Structured Streaming**: Process data as unbounded tables
- **Watermarks**: Handle late-arriving data in windowed aggregations
- **Window Functions**: Group data into time buckets (hourly, daily)
- **foreachBatch**: Apply complex operations (MERGE) in streaming pipelines
- **Checkpointing**: Track processed data for fault tolerance

### Task 3.1: Simulate Streaming Source

In production, trip data arrives as JSON files uploaded hourly from taxi meters. Simulate this by exporting sample trips as 10 JSON file batches (representing 10 hours of data).

In [0]:
# TODO
# Export sample trips as JSON files (10 batches simulating hourly uploads)

# Get sample trips
sample_trips_df = spark.table("samples.nyctaxi.trips").limit(1000)

# Split into 10 batches and write as JSON
for i in range(10):
    batch_df = sample_trips_df.filter(f"hash(tpep_pickup_datetime) % 10 = {i}")
    (batch_df
     .write
     .format("json")  # JSON format
     .mode("overwrite")  # Overwrite mode
     .save(f"{working_dir}/streaming_source/batch_{i}")
    )

print(f"✅ Created 10 JSON batches simulating hourly taxi meter uploads")
print(f"Source location: {working_dir}/streaming_source")

✅ Created 10 JSON batches simulating hourly taxi meter uploads
Source location: /Volumes/nyctaxi_catalog/analytics/workspace/streaming_source


In [0]:
# CHECK YOUR WORK
source_files = dbutils.fs.ls(f"{working_dir}/streaming_source")
assert len(source_files) == 10, "Should have 10 batches"
print("✅ Task 3.1 complete: Streaming source simulated")

✅ Task 3.1 complete: Streaming source simulated


### Task 3.2: Auto Loader - Bronze Layer

Configure **Auto Loader** to incrementally ingest JSON files. Auto Loader:
- Automatically infers schema from JSON
- Tracks processed files (no duplicates)
- Handles schema evolution automatically
- Scales to millions of files

**Free Edition Note**: We use `trigger(availableNow=True)` instead of continuous processing. This processes all available files then stops (batch-style).

In [0]:
# TODO: Configure Auto Loader with schema inference
# Format: cloudFiles (Databricks Auto Loader)
# cloudFiles.format: json
# cloudFiles.schemaLocation: checkpoint location for inferred schema

bronze_stream_df = (spark.readStream
    .format("cloudFiles")  # CloudFiles format
    .option("cloudFiles.format", "json")  # Source file format
    .option("cloudFiles.schemaLocation", f"{checkpoint_dir}/bronze_schema")
    .load(f"{working_dir}/streaming_source")
)

print("Bronze stream schema:")
bronze_stream_df.printSchema()

# Write to Bronze Delta table
bronze_query = (bronze_stream_df
    .writeStream
    .format("delta")  # Delta format
    .outputMode("append")  # Append mode
    .option("checkpointLocation", f"{checkpoint_dir}/bronze")
    .trigger(availableNow=True)  # availableNow=True
    .start(f"{working_dir}/bronze_taxi_trips")
)

# Wait for completion
bronze_query.awaitTermination()
bronze_query.stop()

print("✅ Bronze layer ingestion complete")

Bronze stream schema:
root
 |-- dropoff_zip: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- pickup_zip: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- _rescued_data: string (nullable = true)

✅ Bronze layer ingestion complete


In [0]:
# Verify Bronze layer
bronze_df = spark.read.format("delta").load(f"{working_dir}/bronze_taxi_trips")
print(f"Bronze layer: {bronze_df.count():,} trips ingested")
display(bronze_df.limit(10))

Bronze layer: 559 trips ingested


dropoff_zip,fare_amount,pickup_zip,tpep_dropoff_datetime,tpep_pickup_datetime,trip_distance,_rescued_data
10110,8.0,10103,2016-02-13T21:57:15.000Z,2016-02-13T21:47:53.000Z,1.4,null
10018,17.5,10013,2016-02-10T09:17:43.000Z,2016-02-10T08:54:25.000Z,3.5,null
10002,15.5,10022,2016-02-19T11:25:12.000Z,2016-02-19T11:09:05.000Z,4.0,null
10003,7.5,10110,2016-02-18T10:21:53.000Z,2016-02-18T10:14:15.000Z,1.28,null
10023,19.0,10002,2016-02-10T23:51:48.000Z,2016-02-10T23:32:23.000Z,5.84,null
10065,6.0,10022,2016-02-04T11:26:48.000Z,2016-02-04T11:19:49.000Z,0.83,null
10119,20.0,10280,2016-02-03T19:40:34.000Z,2016-02-03T19:15:04.000Z,4.7,null
10022,12.0,10012,2016-02-21T04:51:49.000Z,2016-02-21T04:40:46.000Z,3.06,null
10028,33.0,11205,2016-02-20T20:21:57.000Z,2016-02-20T19:38:37.000Z,7.3,null
10025,8.0,10026,2016-02-09T15:40:16.000Z,2016-02-09T15:31:27.000Z,1.18,null


In [0]:
# CHECK YOUR WORK
assert bronze_df.count() > 0, "Bronze table should contain data"
print("✅ Task 3.2 complete: Bronze layer created with Auto Loader")

✅ Task 3.2 complete: Bronze layer created with Auto Loader


### Task 3.3: Streaming Aggregation - Silver Layer

Create **hourly metrics by pickup zone** for real-time operations dashboard:
- Trips per hour per zone
- Total revenue per hour per zone
- Average trip distance per hour per zone

**Key Challenge**: Use **windowed aggregation** with **watermark** to handle late-arriving data. Free Edition requires "append" output mode with watermarks for windowed aggregations.

In [0]:
# TODO: Create streaming aggregation with windowing and watermark
# Add watermark for late data handling, group by hourly windows

from pyspark.sql.functions import window, count, sum, avg, col, to_timestamp

# Read Bronze stream and cast timestamp columns
# Note: Auto Loader reads JSON timestamps as strings, so we need to cast them
silver_stream_df = (spark.readStream
    .format("delta")  # Delta format
    .load(f"{working_dir}/bronze_taxi_trips")
    .withColumn("tpep_pickup_datetime", to_timestamp(col("tpep_pickup_datetime")))
    .withColumn("tpep_dropoff_datetime", to_timestamp(col("tpep_dropoff_datetime")))
    .withWatermark("tpep_pickup_datetime", "1 hour")  # Column and watermark interval
    .groupBy(
        window(col("tpep_pickup_datetime"), "1 hour"),  # Column and window duration
        col("pickup_zip")  # Additional grouping column
    )
    .agg(
        count("*").alias("trips_per_hour"),
        sum("fare_amount").alias("total_revenue"),  # Column to sum
        avg("trip_distance").alias("avg_trip_distance")  # Column to average
    )
)

# Write aggregated stream (use "append" mode - required for windowed aggregations in Free Edition)
silver_query = (silver_stream_df
    .writeStream
    .format("delta")  # Delta format
    .outputMode("append")  # Append mode for windowed aggregations
    .option("checkpointLocation", f"{checkpoint_dir}/silver")
    .trigger(availableNow=True)  # availableNow=True
    .start(f"{working_dir}/silver_hourly_metrics")
)

# Wait for completion
silver_query.awaitTermination()
silver_query.stop()

print("✅ Silver layer aggregation complete")

✅ Silver layer aggregation complete


In [0]:
# Verify Silver layer
silver_df = spark.read.format("delta").load(f"{working_dir}/silver_hourly_metrics")
print(f"Silver layer: {silver_df.count():,} hourly zone metrics")
display(silver_df.orderBy("window", "pickup_zip"))

Silver layer: 549 hourly zone metrics


window,pickup_zip,trips_per_hour,total_revenue,avg_trip_distance
"List(2016-02-01T00:00:00.000Z, 2016-02-01T01:00:00.000Z)",10020,1,5.5,1.12
"List(2016-02-01T06:00:00.000Z, 2016-02-01T07:00:00.000Z)",10016,1,10.5,2.6
"List(2016-02-01T13:00:00.000Z, 2016-02-01T14:00:00.000Z)",10012,1,8.0,1.0
"List(2016-02-01T14:00:00.000Z, 2016-02-01T15:00:00.000Z)",10001,1,26.5,6.4
"List(2016-02-01T17:00:00.000Z, 2016-02-01T18:00:00.000Z)",10012,1,5.0,0.9
"List(2016-02-01T20:00:00.000Z, 2016-02-01T21:00:00.000Z)",10014,1,7.5,1.0
"List(2016-02-01T20:00:00.000Z, 2016-02-01T21:00:00.000Z)",10016,1,12.0,2.9
"List(2016-02-02T07:00:00.000Z, 2016-02-02T08:00:00.000Z)",10103,1,13.5,1.8
"List(2016-02-02T08:00:00.000Z, 2016-02-02T09:00:00.000Z)",10044,1,28.0,7.91
"List(2016-02-02T09:00:00.000Z, 2016-02-02T10:00:00.000Z)",10028,1,14.0,2.9


In [0]:
# CHECK YOUR WORK
assert silver_df.count() > 0, "Silver table should contain aggregated metrics"
assert "window" in silver_df.columns, "Should have window column"
assert "trips_per_hour" in silver_df.columns, "Should have aggregated metrics"
print("✅ Task 3.3 complete: Silver layer with windowed aggregations")

✅ Task 3.3 complete: Silver layer with windowed aggregations


### Task 3.4: Streaming MERGE - Gold Layer (Deduplication)

Create the **Gold layer** by deduplicating streaming data using **MERGE** in a `foreachBatch` function. This ensures:
- No duplicate trips (same pickup time + locations)
- Idempotent pipeline (can rerun safely)
- Production-ready data for BI tools

**Pattern**: `foreachBatch` allows you to run arbitrary code (including MERGE) on each streaming micro-batch.

In [0]:
%sql
-- Create Gold table (managed by Unity Catalog)
CREATE TABLE IF NOT EXISTS nyctaxi_catalog.analytics.taxi_trips_gold (
    tpep_pickup_datetime TIMESTAMP,
    tpep_dropoff_datetime TIMESTAMP,
    trip_distance DOUBLE,
    fare_amount DOUBLE,
    pickup_zip INT,
    dropoff_zip INT
) USING DELTA;

In [0]:
# TODO
# Define foreachBatch function to MERGE streaming data

def upsert_to_gold(batch_df, batch_id):
    """
    Upsert function: MERGE streaming batch into Gold table
    - Update if trip already exists (same pickup time + locations)
    - Insert if trip is new
    """
    print(f"Processing batch {batch_id} with {batch_df.count()} records")

    # Register batch as temp view
    batch_df.createOrReplaceTempView("streaming_batch")

    # Execute MERGE
    spark.sql("""
        MERGE INTO nyctaxi_catalog.analytics.taxi_trips_gold AS target
        USING streaming_batch AS source
        ON target.tpep_pickup_datetime = source.tpep_pickup_datetime
           AND target.pickup_zip = source.pickup_zip
           AND target.dropoff_zip = source.dropoff_zip
        WHEN NOT MATCHED THEN
            INSERT (
                tpep_pickup_datetime, tpep_dropoff_datetime,
                trip_distance, fare_amount,
                pickup_zip, dropoff_zip
            )
            VALUES (
                source.tpep_pickup_datetime, source.tpep_dropoff_datetime,
                source.trip_distance, source.fare_amount,
                source.pickup_zip, source.dropoff_zip
            )
    """)

# Apply foreachBatch to streaming pipeline
# Note: Cast timestamps from string to timestamp type
gold_query = (spark.readStream
    .format("delta")  # Delta format
    .load(f"{working_dir}/bronze_taxi_trips")
    .withColumn("tpep_pickup_datetime", to_timestamp(col("tpep_pickup_datetime")))
    .withColumn("tpep_dropoff_datetime", to_timestamp(col("tpep_dropoff_datetime")))
    .writeStream
    .foreachBatch(upsert_to_gold)  # Function name for batch processing
    .option("checkpointLocation", f"{checkpoint_dir}/gold")
    .trigger(availableNow=True)  # availableNow=True
    .start()
)

# Wait for completion
gold_query.awaitTermination()
gold_query.stop()

print("✅ Gold layer upsert complete")

✅ Gold layer upsert complete


In [0]:
# Verify Gold layer
gold_df = spark.table("nyctaxi_catalog.analytics.taxi_trips_gold")
print(f"Gold layer: {gold_df.count():,} unique trips")
display(gold_df.limit(10))

Gold layer: 559 unique trips


tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110
2016-02-10T08:54:25.000Z,2016-02-10T09:17:43.000Z,3.5,17.5,10013,10018
2016-02-19T11:09:05.000Z,2016-02-19T11:25:12.000Z,4.0,15.5,10022,10002
2016-02-18T10:14:15.000Z,2016-02-18T10:21:53.000Z,1.28,7.5,10110,10003
2016-02-10T23:32:23.000Z,2016-02-10T23:51:48.000Z,5.84,19.0,10002,10023
2016-02-04T11:19:49.000Z,2016-02-04T11:26:48.000Z,0.83,6.0,10022,10065
2016-02-03T19:15:04.000Z,2016-02-03T19:40:34.000Z,4.7,20.0,10280,10119
2016-02-21T04:40:46.000Z,2016-02-21T04:51:49.000Z,3.06,12.0,10012,10022
2016-02-20T19:38:37.000Z,2016-02-20T20:21:57.000Z,7.3,33.0,11205,10028
2016-02-09T15:31:27.000Z,2016-02-09T15:40:16.000Z,1.18,8.0,10026,10025


In [0]:
# CHECK YOUR WORK
assert gold_df.count() > 0, "Gold table should contain data"
# Count should be <= Bronze count (duplicates removed)
bronze_count = spark.read.format("delta").load(f"{working_dir}/bronze_taxi_trips").count()
assert gold_df.count() <= bronze_count, "Gold should have same or fewer records (duplicates removed)"
print("✅ Task 3.4 complete: Gold layer with streaming MERGE deduplication")

✅ Task 3.4 complete: Gold layer with streaming MERGE deduplication


---
# Conclusion

Congratulations! You've built a complete **Delta Lake data pipeline** for NYC taxi analytics:

## What You've Accomplished:

✅ **Delta Lake Foundations** - Delta format, schema evolution, time travel, table history
✅ **Data Management Operations** - CRUD operations, MERGE upserts, OPTIMIZE, Z-ordering, VACUUM
✅ **Streaming Pipeline** - Auto Loader, Bronze → Silver → Gold architecture, windowed aggregations, streaming MERGE

## Key Takeaways:

1. **Delta Lake = ACID on Data Lakes** - Get database features (transactions, updates, deletes) on cheap object storage
2. **Schema Evolution is Flexible** - Add columns with `mergeSchema`, change types with `overwriteSchema`
3. **Time Travel is Powerful** - Audit changes, rollback mistakes, reproduce ML datasets
4. **MERGE Simplifies Upserts** - Update + insert in one atomic operation
5. **OPTIMIZE + Z-Order = Fast Queries** - Compact files and cluster related data together
6. **Auto Loader = Scalable Ingestion** - Process millions of files with schema inference
7. **Medallion Architecture = Best Practice** - Bronze (raw) → Silver (cleaned) → Gold (aggregated)
8. **Free Edition is Production-Ready** - Same APIs as full Databricks, just with batch triggers

## Production Considerations:

- **Retention**: Use `VACUUM RETAIN 168 HOURS` (7 days) in production to preserve time travel
- **Z-Order**: Choose columns used in WHERE clauses (not necessarily primary keys)
- **OPTIMIZE Frequency**: Run daily on hot tables, weekly on cold tables
- **Checkpoints**: Never delete checkpoint directories (breaks streaming pipelines)
- **Watermarks**: Balance between latency (small watermark) and completeness (large watermark)

You've mastered Delta Lake management and streaming - essential skills for modern data engineering!

## Cleanup

In [0]:
# Stop all active streaming queries
for query in spark.streams.active:
    print(f"Stopping query: {query.id}")
    query.stop()

In [0]:
# Optional: Clean up all lab data (uncomment to execute)
# dbutils.fs.rm(working_dir, recurse=True)
# spark.sql("DROP TABLE IF EXISTS nyctaxi_catalog.analytics.taxi_trips")
# spark.sql("DROP TABLE IF EXISTS nyctaxi_catalog.analytics.taxi_trips_gold")
# print("✅ Lab cleanup complete")